# 01 – Image Check (Baselines / Evaluation – Sophia)

Goal of this notebook: load one patient, check sizes and value ranges, and look at the images
before building the FBP baseline and the metrics.

**Data (Souram's output on Google Drive):**
- `full_output/series_XXXX/slice_NN.png` – full-view FBP (984 views) = ground truth
- `sparse_output/series_XXXX/slice_NN.png` – sparse-view FBP

Before running: add both shared folders to "My Drive" as shortcuts (Organise → Add shortcut)
and adjust `DATA_ROOT` below if your folder is named differently.

## 0. Setup

In [1]:
# Mount Google Drive (data is NOT in git – too large)
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = "/content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction"   # <- adjust if needed

ModuleNotFoundError: No module named 'google'

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

FULL_DIR   = Path(DATA_ROOT) / "full_output"     # ground truth (full-view FBP)
SPARSE_DIR = Path(DATA_ROOT) / "sparse_output"   # sparse-view FBP
print("full series:  ", len(list(FULL_DIR.glob("series_*"))))
print("sparse series:", len(list(SPARSE_DIR.glob("series_*"))))

## 1. Which patients and slices do we have?

In [ ]:
def slice_index(path):
    # 'slice_07.png' -> 7  (sort numerically, not alphabetically!)
    return int(re.search(r"(\d+)", path.stem).group(1))

def list_slices(folder, pattern):
    return sorted(folder.glob(pattern), key=slice_index)

rows = []
for d in sorted(SPARSE_DIR.glob("series_*")):
    sid = d.name
    sparse_names = {f.name for f in d.glob("*.png")}
    full_names   = {f.name for f in (FULL_DIR / sid).glob("*.png")} if (FULL_DIR / sid).exists() else set()
    rows.append({"series": sid, "n_full": len(full_names), "n_sparse": len(sparse_names),
                 "n_paired": len(full_names & sparse_names),        # same file name in both folders
                 "all_paired": full_names == sparse_names})
overview = pd.DataFrame(rows)
overview

## 2. Loader functions

In [ ]:
def to_gray(arr):
    # RGB(A) uint8 -> 2D float array in [0, 1]. Checks first that R, G, B are identical (pure greyscale).
    if arr.ndim == 3:
        rgb = arr[..., :3].astype(np.int16)
        max_diff = np.abs(rgb - rgb[..., :1]).max()
        if max_diff > 2:   # JPG compression can cause tiny differences
            print(f"  warning: colour channels differ by up to {max_diff}")
        arr = arr[..., 0]
    return arr.astype(np.float32) / 255.0

def load_patient(folder, pattern):
    # Returns a 3D stack (slices, H, W) as float32 in [0, 1]
    files = list_slices(folder, pattern)
    return np.stack([to_gray(np.array(Image.open(f))) for f in files])

PATIENT = overview.series.iloc[0]                 # first series; change to look at others
orig   = load_patient(FULL_DIR / PATIENT, "*.png")
sparse = load_patient(SPARSE_DIR / PATIENT, "*.png")
print("full:    ", orig.shape, orig.dtype)
print("sparse:  ", sparse.shape, sparse.dtype)

## 3. Sizes and value ranges

In [ ]:
# Raw file properties (mode = RGB / RGBA / L, bit depth)
for name, f in [("full", list_slices(FULL_DIR / PATIENT, "*.png")[0]),
                ("sparse",   list_slices(SPARSE_DIR / PATIENT, "*.png")[0])]:
    im = Image.open(f)
    a = np.array(im)
    print(f"{name:8s} {f.name:14s} mode={im.mode:5s} shape={a.shape} dtype={a.dtype}")
    if im.mode == "RGBA":
        print(f"         alpha channel unique values: {np.unique(a[..., 3])[:5]}")

In [ ]:
def stats(stack, name):
    return {"data": name, "shape": stack.shape, "min": stack.min(), "max": stack.max(),
            "mean": stack.mean().round(3), "std": stack.std().round(3),
            "unique_grey_levels": len(np.unique(stack))}

pd.DataFrame([stats(orig, "full (GT)"), stats(sparse, "sparse")])

## 4. Look at the images

In [ ]:
idx = [0, len(sparse) // 2, len(sparse) - 1]   # first, middle, last slice
fig, ax = plt.subplots(2, len(idx), figsize=(4 * len(idx), 8))
for j, i in enumerate(idx):
    ax[0, j].imshow(orig[i],   cmap="gray"); ax[0, j].set_title(f"full-view FBP – slice {i}")
    ax[1, j].imshow(sparse[i], cmap="gray"); ax[1, j].set_title(f"sparse FBP – slice {i}")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Intensity histograms (background pixels dominate, so log scale on y)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(orig.ravel(), bins=256);   ax[0].set_title("full-view FBP");   ax[0].set_yscale("log")
ax[1].hist(sparse.ravel(), bins=256); ax[1].set_title("sparse FBP"); ax[1].set_yscale("log")
for a in ax: a.set_xlabel("intensity [0–1]")
plt.tight_layout(); plt.show()

## 5. How much of the image is head vs. background?

In [ ]:
# Relevant for the metrics: a large black background inflates PSNR/SSIM.
THRESH = 0.05
for name, s in [("full", orig), ("sparse", sparse)]:
    frac = (s > THRESH).mean()
    print(f"{name:8s}: {100 * frac:.1f}% of pixels above {THRESH}")

## Findings / notes for the log
- Do full and sparse have the same size / slice count? (section 1 + 2)
- Value range and number of grey levels (8-bit PNG → max. 256 levels)?
- Which view count do the sparse images use? (ask Souram – not in the file name)
- To do: …